## Building Multi Step Workflows

This looks like a great lesson on orchestrating serverless microservices! I’ve cleaned up the formatting using proper Markdown headers, code blocks, and visual cues to make the flow easier for students to follow.

---

# Introduction: The Need for Multi-Step Workflows

Welcome back! In the last lesson, you learned how to build and deploy simple serverless APIs using Google Cloud Functions and API Gateway. You saw how a single Cloud Function can handle a request and return a response.

However, many real-world applications require more than just a single step. For example, when processing an online order, you might need to:

1.  **Validate** the order details.
2.  **Charge** the customer.
3.  **Record** the order in a database.



Each of these steps could be handled by a separate Cloud Function. But how do you make sure they run in the right order and that data flows smoothly from one step to the next? This is where **Google Cloud Workflows** come in. Workflows let you organize and manage multi-step processes in your serverless applications.

---

## Quick Recall: Cloud Functions in Serverless Apps

Before we dive into Workflows, let's quickly remind ourselves how Cloud Functions fit into serverless applications. A **Cloud Function** is a small piece of code that runs in response to an event, such as an API call or a message. 

In the previous lesson, you learned how to:
* Write a Cloud Function that processes input and returns output.
* Connect a Cloud Function to an API Gateway endpoint.

In this lesson, we will see how to connect several Cloud Functions together to handle a more complex process.

---

## Understanding Google Cloud Workflows

Google Cloud Workflows help you coordinate multiple Cloud Functions (or other Google Cloud services) into a single workflow. A workflow is like a **flowchart**: it defines a series of steps and the order in which they run. Each step can perform a task, make a choice, or wait for something to happen.

### The Order Processing Example
Imagine you have three steps:
1.  **Validate the order** (check if the amount is valid).
2.  **Charge the customer** (simulate payment).
3.  **Record the order** (save it to a database).

With Workflows, you can define a process that runs these steps one after another, passing the result from one step to the next.

---

## Breaking Down the Order Workflow Example

Let's build up the order processing workflow step-by-step using three Cloud Functions.

### Step 1: Validate the Order
First, we need a Cloud Function to check if the order amount is valid.

```python
# main.py (validate_order function)
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}
```

* **Logic:** If the amount is valid, it adds a `"validated": True` field.
* **Input Example:** `{"order": {"order_id": "test-123", "amount": "99.99"}}`
* **Output Example:** `{"order": {"order_id": "test-123", "amount": "99.99", "validated": true}}`

### Step 2: Charge the Customer
Next, we simulate charging the customer using the output from Step 1.

```python
# main.py (charge_customer function)
def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
```

* **Output Example:** `{"order": {...}, "payment_id": "pay_test-123", "charged": true}`

### Step 3: Record the Order
Finally, we save the order to a **Firestore** collection.

```python
# main.py (record_order function)
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}
```

---

## Defining the Workflow (`YAML`)

This `order_workflow.yaml` file acts as the "glue" that connects our functions.

```yaml
main:
  params: [input]
  steps:
    - validate:
        call: http.post
        args:
          url: https://REGION-PROJECT_ID.cloudfunctions.net/validate_order
          body: ${input}
        result: validated
    - charge:
        call: http.post
        args:
          url: https://REGION-PROJECT_ID.cloudfunctions.net/charge_customer
          body: ${validated.body}
        result: charged
    - record:
        call: http.post
        args:
          url: https://REGION-PROJECT_ID.cloudfunctions.net/record_order
          body: ${charged.body}
        result: recorded
    - return_output:
        return: ${recorded.body}
```

---

## Deploying and Testing

### 1. Deploy Cloud Functions
```bash
gcloud functions deploy validate_order --runtime python310 --trigger-http --allow-unauthenticated
gcloud functions deploy charge_customer --runtime python310 --trigger-http --allow-unauthenticated
gcloud functions deploy record_order --runtime python310 --trigger-http --allow-unauthenticated
```

### 2. Deploy the Workflow
```bash
gcloud workflows deploy order-workflow --source=order_workflow.yaml --location=REGION
```

### 3. Execute the Workflow
Trigger the workflow with sample data:
```bash
gcloud workflows execute order-workflow \
  --data='{"order": {"order_id": "test-123", "amount": "99.99"}}' \
  --location=REGION
```

**Expected Final Output:**
```json
{
  "status": "ok", 
  "order": {
    "order_id": "test-123", 
    "amount": 99.99, 
    "payment_id": "pay_test-123", 
    "ts": 1710000000
  }
}
```

---

## Summary and Practice Preview

In this lesson, you learned how to:
* **Deconstruct** a complex process into modular Cloud Functions.
* **Orchestrate** those modules using a YAML-based workflow.
* **Manage State** by passing variables from one step to the next.

Next, you'll get hands-on practice building your own Workflows! Good luck!

## Fixing Order Amount Validation in Cloud Function

You have a multi-step order processing workflow using Google Cloud Workflows and Cloud Functions. There is a bug in the validation step: the system is currently accepting orders with zero amounts as valid, which should not happen.

Your task is to examine the validate_order function and fix the conditional statement so that orders with exactly zero amounts are properly rejected.

Test your fix with these cases:

    Orders with negative amounts should be rejected (already working)
    Orders with zero amounts should be rejected (currently broken)
    Orders with positive amounts should be accepted (already working)

Debugging and fixing validation logic is essential for building reliable serverless workflows.

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) < 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}
    
def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
    
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}

```

The issue in your `validate_order` function lies in the comparison operator. Currently, it only checks if the amount is strictly less than zero ($< 0$), which allows $0$ to slip through as a valid value.

To fix this, you should change the condition to check if the amount is **less than or equal to zero** ($\le 0$).

### Fixed Code

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    
    # FIX: Changed < to <= to reject zero-amount orders
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
        
    order["validated"] = True
    return {"order": order}
```

---

### Why this fix works:

* **Negative Amounts (-5.00):** These are still caught because $-5 \le 0$ is **True**.
* **Zero Amounts (0.00):** These are now caught because $0 \le 0$ is **True**.
* **Positive Amounts (10.00):** These are still accepted because $10 \le 0$ is **False**, allowing the function to proceed to set `order["validated"] = True`.

### Pro-Tip for Production
When dealing with financial data, it's also a good idea to wrap your `float()` conversion in a `try/except` block. If the input is missing or is a non-numeric string (like `"abc"`), the function will crash with a `500 Internal Server Error` instead of returning a clean `400 Bad Request` error message to your workflow.

## Preserving Data Fields Across Workflow Steps in Cloud Functions

Now that you understand how Google Cloud Workflows coordinate multiple Cloud Functions, it's time to practice maintaining data flow across your workflow steps. Currently, our workflow handles basic order information, but we need to add support for tracking customer_email addresses throughout the entire process.

Your objective is to modify all three Cloud Functions to handle a new customer_email field that will be included in the original order input. Here's what you need to do:

    Update the validate_order function to preserve the customer_email field in the order object
    Update the charge_customer function to keep the customer_email field in the request data
    Update the record_order function to store the customer_email in the Firestore document

The key concept here is understanding how data moves from one step to the next in a workflow. Each function receives the output of the previous function as its input, so you need to make sure important data isn't lost along the way.

Test your workflow with input like this:

JSON
{"order": {"order_id": "test-123", "amount": "99.99", "customer_email": "customer@example.com"}}

When everything works correctly, you should see the customer email stored in your Firestore collection alongside the order details. This exercise will help you master data continuity in serverless workflows — a skill you'll use in every real-world Cloud Workflows application.

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    # TODO: Make sure the customer_email field is preserved in the order object
    return {"order": order}

def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    # TODO: Ensure all data including customer_email flows through to the next step
    return request_json

import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    # TODO: Add the customer_email field to the Firestore document
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}
```

## Adding Error Handling to a Google Cloud Workflow

## Adding a Notification Step to a Google Cloud Workflow

## Building a Multi-Step Returns Workflow